In [3]:
import sys
import os
import s3fs
import warnings
from pathlib import Path
import xarray as xr

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.config_utils import check_runtime_readiness, load_pipeline_config, resolve_secrets
from pipeline.inventory import build_inventory_snapshot_and_diff
from pipeline.generate_parquet import concurrent_dask_ref_generation, save_ledger_after_success, _build_registry

import virtualizarr as vz
from virtualizarr.parsers import HDFParser
from obstore.store import S3Store
from obspec_utils.registry import ObjectStoreRegistry
import traceback
print('Imports OK')

SyntaxError: invalid syntax (generate_parquet.py, line 111)

In [5]:
check_runtime_readiness()
kp= load_pipeline_config("../configs/config.yaml")
ACCESS_KEY, SECRET_KEY = resolve_secrets(kp)

warnings.filterwarnings(
  "ignore",
  message="Numcodecs codecs are not in the Zarr version 3 specification*",
  category=UserWarning
)

In [ ]:
out_path = Path("../.tmp/DPIRD_final_stations.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)

key = "FINAL_DPIRD/DPIRD_final_stations.nc"
source_url = f"s3://{kp['s3']['bucket']}/{key}"
registry = _build_registry(kp, ACCESS_KEY, SECRET_KEY)

string_coords= ['station', 'code']
parser= HDFParser(drop_variables=string_coords)     # THIS LINE IS THE FIX
label = parser.__class__.__name__

try:
    vds = vz.open_virtual_dataset(
        url=source_url,
        registry=registry,
        parser=parser,
    )
    
    # Mount the S3 file to eagerly load the missing 1D strings 
    fs = s3fs.S3FileSystem(
        key=ACCESS_KEY, 
        secret=SECRET_KEY, 
        client_kwargs={"endpoint_url": kp["s3"]["endpoint_url"]},
        config_kwargs={"signature_version": "s3v4", "s3": {"addressing_style": "path"}}
    )
    with fs.open(f"{kp['s3']['bucket']}/{key}", "rb") as f:
        ds_real = xr.open_dataset(f, engine="h5netcdf")
        
        for var in string_coords:
            if var in ds_real:
                vds = vds.assign_coords({var: ds_real[var].load()})
                
    # Export to Parquet. The lazy arrays become byte-range references; 
    vds.vz.to_kerchunk(
        filepath=str(out_path), 
        format="parquet",
        record_size=100000,
        categorical_threshold=10,
    )
    print("PASS:", label, "->", out_path)

except Exception as exc:
    print("FAIL:", label, "->", type(exc).__name__, str(exc))
    traceback.print_exc()

PASS: HDFParser -> ..\.tmp\DPIRD_final_stations.parquet


In [6]:
import fsspec

storage_options= {
    "key":ACCESS_KEY, 
    "secret":SECRET_KEY, 
    "client_kwargs":{"endpoint_url": kp["s3"]["endpoint_url"]},
    "config_kwargs":{"signature_version": "s3v4", "s3": {"addressing_style": "path"}},
}

local_parquet_path= "../.tmp/DPIRD_final_stations.parquet"
try:
    ds_check= xr.open_dataset(
        local_parquet_path,
        engine= 'kerchunk',
        backend_kwargs= {"storage_options": {"remote_options": storage_options}}
    )
    print(ds_check)
except Exception as e:
    print(f"Could not open virtual dataset: {e}")
    traceback.print_exc()

<xarray.Dataset> Size: 3GB
Dimensions:                       (station: 192, time: 105248)
Coordinates:
  * station                       (station) <U22 17kB 'Floreat Park' ... 'Gut...
    code                          (station) <U5 4kB ...
    lat                           (station) float64 2kB ...
    lon                           (station) float64 2kB ...
  * time                          (time) datetime64[ns] 842kB 2022-01-01T08:0...
Data variables: (12/18)
    airTemperature                (station, time) float64 162MB ...
    apparentAirTemperature        (station, time) float64 162MB ...
    deltaT                        (station, time) float64 162MB ...
    dewPoint                      (station, time) float64 162MB ...
    evapotranspiration_shortCrop  (station, time) float64 162MB ...
    evapotranspiration_tallCrop   (station, time) float64 162MB ...
    ...                            ...
    solarExposure                 (station, time) float64 162MB ...
    wetBulb         

In [2]:
import logging

# 1. Configure logging to show HTTP requests made by s3fs/botocore
s3_logger = logging.getLogger("s3fs")
s3_logger.setLevel(logging.DEBUG)

# Ensure logs appear in the notebook
if not s3_logger.handlers:
    sh = logging.StreamHandler()
    sh.setFormatter(logging.Formatter('%(levelname)s:%(name)s:%(message)s'))
    s3_logger.addHandler(sh)

print("--- Accessing 'lat' coordinate (Small read) ---")
# Accessing values triggers the kerchunk reference lookup and S3 fetch
lats = ds_check.lat.values[:5] 
print(f"Lat values: {lats}")

print("\n--- Accessing a slice of 'airTemperature' (Specific byte range) ---")
# This variable is large, so we only fetch a tiny chunk to prove range requests work
temp_slice = ds_check.airTemperature.isel(station=0, time=slice(0, 10)).values
print(f"Temp slice: {temp_slice}")

# 2. Reset logging to avoid cluttering later cells
s3_logger.setLevel(logging.INFO)

--- Accessing 'lat' coordinate (Small read) ---


NameError: name 'ds_check' is not defined

In [15]:
#list(ds_check.variables)
ds_check.code.dtype.kind

'U'